In [1]:
import os
import pickle
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from torch_geometric.loader import NeighborLoader
from torch_geometric.utils import to_undirected
from torch_geometric.utils.convert import from_networkx
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR

from dotenv import load_dotenv

warnings.filterwarnings("ignore")


In [ ]:
# current dir
notebooks_dir = Path().resolve()
# repo root
repo_root = notebooks_dir.parent.parent
# src
src_path = repo_root / "src"
# bsard data path
bsard_data_path = repo_root / "data" / "BSARD_dataset"

# insert src in sys.path for importing modules
sys.path.insert(0, str(src_path))

In [3]:
###############
# BASE DOCUMENT GRAPH BUILDING
###############

from BSARD.doc_graph_building import build_document_graph

# Load inputs
bsard_corpus_path = 'inputs/bsard_corpus.csv'
bsard_corpus = pd.read_csv(os.path.join(bsard_data_path, bsard_corpus_path))

# Execute main
G_base, bsard_corpus_lean = build_document_graph(bsard_corpus)

# Save outputs
with open(os.path.join(bsard_data_path, 'intermediate', "G_base.pkl"), 'wb') as f:
    pickle.dump(G_base, f)

bsard_corpus_lean.to_csv(os.path.join(bsard_data_path, 'intermediate', "bsard_corpus_processed.csv"))

In [4]:
###############
# HYBRID GRAPH BUILDING
###############

from BSARD.hybrid_graph_building import build_hybrid_graph

# Define key values
keyterm_mincount = 5
unique_act_mincount = 2

# Load inputs
G_base_path = os.path.join(bsard_data_path, 'intermediate', 'G_base.pkl')
with open(G_base_path, 'rb') as f:
    G_base = pickle.load(f)

keyterms_dict_path = os.path.join(bsard_data_path, 'inputs', 'keyterms_dict.pkl')
with open(keyterms_dict_path, 'rb') as f:
    keyterms_dict = pickle.load(f)

# Execute main
G_hybrid = build_hybrid_graph(G_base, keyterms_dict, keyterm_mincount=keyterm_mincount, unique_act_mincount=unique_act_mincount)

# Save outputs
with open(os.path.join(bsard_data_path, 'intermediate', "G_hybrid.pkl"), 'wb') as f:
    pickle.dump(G_hybrid, f)

In [ ]:
###############
# SEMANTIC EMBEDDING GENERATON
###############

from BSARD.semantic_embedding_genration import assign_semantinc_node_embeddings

# Define key values
load_dotenv(os.path.join(repo_root, '.env'))
NGROK_URL = os.getenv("NGROK_URL")
API_TOKEN = os.getenv("API_KEY")

# Load inputs
with open(os.path.join(bsard_data_path, "intermediate", "G_hybrid.pkl"), 'rb') as f:
    G_hybrid = pickle.load(f)

# Execute main
G_semantic = assign_semantinc_node_embeddings(G_hybrid, url=NGROK_URL, api_token=API_TOKEN)

# Ensure all nodes have an attribute "embedding"
assert all("embedding" in data for _, data in G_semantic.nodes(data=True)), "Some nodes lack an embedding!"

# Save outputs
with open(os.path.join(bsard_data_path, 'intermediate/G_semantic.pkl'), 'wb') as f:
    pickle.dump(G_semantic, f)

Generating Central Node embedding... : 100%|██████████| 1/1 [00:00<00:00, 1296.94it/s]


In [ ]:
###############
# SEMANTIC EDGES
###############

from BSARD.semantic_edges import add_semantic_edges

# Define key values
n_similarity_edges = 10

# Load inputs
with open(os.path.join(bsard_data_path, "intermediate", "G_hybrid.pkl"), 'rb') as f:
    G_hybrid = pickle.load(f)

# Execute main
G_pretraining = add_semantic_edges(G_hybrid, n=n_similarity_edges)

# Save outputs
with open(os.path.join(bsard_data_path, 'intermediate/G_pretraining.pkl'), 'wb') as f:
    pickle.dump(G_semantic, f)